In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
print('Setup complete')

Setup complete


In [0]:
display(dbutils.fs.ls('/Volumes/filestore/olist/data'))

path,name,size,modificationTime
dbfs:/Volumes/filestore/olist/data/archive/,archive/,0,1775726995632
dbfs:/Volumes/filestore/olist/data/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1775641228000
dbfs:/Volumes/filestore/olist/data/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1775641248000
dbfs:/Volumes/filestore/olist/data/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1775641230000
dbfs:/Volumes/filestore/olist/data/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1775641226000
dbfs:/Volumes/filestore/olist/data/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1775641230000
dbfs:/Volumes/filestore/olist/data/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1775641231000
dbfs:/Volumes/filestore/olist/data/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1775641224000
dbfs:/Volumes/filestore/olist/data/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1775641222000
dbfs:/Volumes/filestore/olist/data/product_category_name_translation.csv,product_category_name_translation.csv,2613,1775641222000


In [0]:
customers = spark.read.csv('/Volumes/filestore/olist/data/olist_customers_dataset.csv', header=True, inferSchema=True)
orders = spark.read.csv('/Volumes/filestore/olist/data/olist_orders_dataset.csv', header=True, inferSchema=True)
order_items = spark.read.csv('/Volumes/filestore/olist/data/olist_order_items_dataset.csv', header=True, inferSchema=True)
products = spark.read.csv('/Volumes/filestore/olist/data/olist_products_dataset.csv', header=True, inferSchema=True)
payments = spark.read.csv('/Volumes/filestore/olist/data/olist_order_payments_dataset.csv', header=True, inferSchema=True)

In [0]:
#Task 1: Top 3 Customers per City Calculate total spend per customer, then use window function to rank customers within each city. Output: city, customer_id, total_spend, rank
spend_df = customers.join(orders, "customer_id").join(order_items, "order_id").groupBy("customer_city", "customer_id").agg(sum("price").alias("total_spend"))
window_spec = Window.partitionBy("customer_city").orderBy(col("total_spend").desc())
spend_df.withColumn("rank", rank().over(window_spec)).filter(col("rank") <= 3).show()



+-------------------+--------------------+-----------+----+
|      customer_city|         customer_id|total_spend|rank|
+-------------------+--------------------+-----------+----+
|abadia dos dourados|9e01f714a2b3b8962...|      199.0|   1|
|abadia dos dourados|a23e3f9a2b656b23b...|      120.0|   2|
|abadia dos dourados|f11eb8f0b8b87510a...|       39.9|   3|
|          abadiania|576d71ddb21b21763...|     949.99|   1|
|             abaete|d47c8bb51df6f716e...|      449.0|   1|
|             abaete|ff0d62f8be4c098e6...|      225.9|   2|
|             abaete|5371894984937a27c...|      208.9|   3|
|         abaetetuba|c7eb06383ae604616...|     1500.0|   1|
|         abaetetuba|367fd22f1e994de87...|      797.6|   2|
|         abaetetuba|7727e2cc9ec428ad3...|     580.27|   3|
|            abaiara|f8508e9ec506a046a...|      169.0|   1|
|            abaiara|9723d86b12bdec025...|       93.9|   2|
|             abaira|11888d3334232a1eb...|      145.0|   1|
|             abaira|43b0018187e283119..

In [0]:
#Task 2: Running Total of Sales Calculate daily sales and then cumulative (running) total using window function. Output: date, daily_sales, running_total

daily_sales = orders.join(order_items, "order_id").withColumn("date", to_date("order_purchase_timestamp")).groupBy("date").agg(sum("price").alias("daily_sales"))
window_spec = Window.orderBy("date")
result = daily_sales.withColumn("running_total", sum("daily_sales").over(window_spec)).show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+------------------+------------------+
|      date|       daily_sales|     running_total|
+----------+------------------+------------------+
|2016-09-04|             72.89|             72.89|
|2016-09-05|              59.5|            132.39|
|2016-09-15|            134.97|            267.36|
|2016-10-02|             100.0|            367.36|
|2016-10-03|            463.48|            830.84|
|2016-10-04|           9940.96|           10771.8|
|2016-10-05|           8343.25|          19115.05|
|2016-10-06|           7960.51|27075.559999999998|
|2016-10-07|           7228.05|          34303.61|
|2016-10-08| 8441.849999999999|          42745.46|
|2016-10-09|3336.9900000000002|          46082.45|
|2016-10-10|3692.5699999999997|          49775.02|
|2016-12-23|              10.9|          49785.92|
|2017-01-05|396.90000000000003|          50182.82|
|2017-01-06|            916.38|           51099.2|
|2017-01-07|            1351.9|           52451.1|
|2017-01-08|            709.58|

In [0]:
#Task 3: Top Products per Category Aggregate sales per product, join with category, then rank using DENSE_RANK. Output: category, product_id, total_sales, rank

payments_agg = payments.groupBy("order_id").agg(_sum("payment_value").alias("total_payment"))
base_df = order_items \
    .join(payments_agg, "order_id") \
    .join(products, "product_id")


base_df = base_df.filter(col("product_category_name").isNotNull())

product_sales = base_df.groupBy("product_id", "product_category_name") \
    .agg(_sum("total_payment").alias("total_sales"))

window_spec = Window.partitionBy("product_category_name") \
    .orderBy(col("total_sales").desc())

ranked_products = product_sales.withColumn(
    "rank",
    dense_rank().over(window_spec)
)

display(ranked_products.select(
    col("product_category_name"),
    "product_id",
    "total_sales",
    "rank"
))

product_category_name,product_id,total_sales,rank
agro_industria_e_comercio,c183fd5d2abf05873fa6e1014ed9e06c,36489.24,1
agro_industria_e_comercio,11250b0d4b709fee92441c5f34122aed,10589.920000000002,2
agro_industria_e_comercio,672e757f331900b9deea127a2a7b79fd,9927.779999999999,3
agro_industria_e_comercio,423a6644f0aa529e8828ff1f91003690,8609.73,4
agro_industria_e_comercio,2b69866f22de8dad69c976771daba91c,3184.55,5
agro_industria_e_comercio,c89226b8a795ae3d6bca9d90b20dbf04,2893.49,6
agro_industria_e_comercio,5fb0955cb683eb6f65a1f613e502eef5,2776.1099999999997,7
agro_industria_e_comercio,b7a60a397d4efd05c1b5d398fb9f9097,2467.33,8
agro_industria_e_comercio,cd5df6a3db7a3d064a55afd08289d762,2398.36,9
agro_industria_e_comercio,cd2f5c10e4e8dbc701f0bb68a09fdfe8,2315.83,10


In [0]:
#Task 4: Customer Lifetime Value Calculate total spend per customer across all orders. Output: customer_id, total_spend
orders.join(order_items, "order_id").groupBy("customer_id").agg(sum("price").alias("total_spend")).show()

+--------------------+-----------+
|         customer_id|total_spend|
+--------------------+-----------+
|cf8ffeddf027932e5...|      179.0|
|241e78de29b3090cf...|      113.0|
|e1862648f338ecaf4...|      37.99|
|3c628393675b42c6b...|      359.8|
|328d7a69cb9cbaf08...|       90.0|
|aa601b3c45980c091...|      49.99|
|d4c5a2a1316f738e7...|       96.0|
|ae8db0691449a4435...|      109.9|
|4d225460f83ea2b1a...|      89.99|
|af5b08db5f1a31a3b...|      39.99|
|1ae196062dab95e43...|       39.0|
|fbf8e675c5adcf8c4...|      109.9|
|5613454932b2c4229...|       40.0|
|94e56a6d6f8b3e26a...|      119.0|
|f31eb05ebdef1ede7...|      170.0|
|b7919647bde69acc9...|      168.0|
|638c6674418fc5828...|      225.0|
|cf3c35f463f6a234d...|      109.8|
|661652a4ba9255d6a...|       24.9|
|5ed8d20445c356efa...|      32.64|
+--------------------+-----------+
only showing top 20 rows


In [0]:
#Task 5: Customer Segmentation Apply logic: total_spend > 10000 → Gold 5000–10000 → Silver <5000 → Bronze Add a new column 'segment'. Then group by segment to count customers. Output: customer_id, total_spend, segment

segmented = orders.join(order_items, "order_id").groupBy("customer_id") \
        .agg(sum("price").alias("total_spend")).withColumn(
    "segment",
    when(col("total_spend") > 10000, "Gold")
    .when(col("total_spend") >= 5000, "Silver")
    .otherwise("Bronze")
)

segment_count=segmented.groupBy("segment").count()

display(segmented)
display(segment_count)

customer_id,total_spend,segment
cf8ffeddf027932e51e4eae73b384059,179.0,Bronze
241e78de29b3090cfa1b5d73a8130c72,113.0,Bronze
e1862648f338ecaf4242e8e2b59126ca,37.99,Bronze
3c628393675b42c6b5ef89461f68ecef,359.8,Bronze
328d7a69cb9cbaf088eed3ed778804bb,90.0,Bronze
aa601b3c45980c0918042d5ca7a25054,49.99,Bronze
d4c5a2a1316f738e72e179d6bb3b6d8f,96.0,Bronze
ae8db0691449a44352e7d535ddf78c5e,109.9,Bronze
4d225460f83ea2b1ac705ad1d5eef02f,89.99,Bronze
af5b08db5f1a31a3b6856a8a6dc33167,39.99,Bronze


segment,count
Bronze,98660
Silver,5
Gold,1


In [0]:
#Task 6: Final Reporting Table Combine results into a single dataset containing: customer_id, city, total_spend, segment, total_orders
orders_count = orders.groupBy("customer_id").count().withColumnRenamed("count", "total_orders")

customers.join(segmented, "customer_id").join(orders_count, "customer_id").select("customer_id", "customer_city", "total_spend", "segment", "total_orders").show()

+--------------------+--------------------+-----------+-------+------------+
|         customer_id|       customer_city|total_spend|segment|total_orders|
+--------------------+--------------------+-----------+-------+------------+
|816f8653d5361cbf9...|           sao paulo|       49.0| Bronze|           1|
|a9d37ddc8ba4d9f6d...|           joinville|       27.9| Bronze|           1|
|bb2f5e670f7155dc6...|           sao paulo|       50.0| Bronze|           1|
|388025bec8128ff20...|            cascavel|      190.0| Bronze|           1|
|818596f5b68adfe2c...|      rio de janeiro|       63.6| Bronze|           1|
|4d225460f83ea2b1a...|           cariacica|      89.99| Bronze|           1|
|0fa5382628eea6bbb...|      alfredo chaves|      22.32| Bronze|           1|
|d1c6e90ab127d42b7...|         avanhandava|     179.99| Bronze|           1|
|d29935fdaf4a76f34...|            confresa|      189.9| Bronze|           1|
|2fdffca8dcdf01547...|             itatiba|      109.0| Bronze|           1|